<a href="https://colab.research.google.com/github/Shreshth1805/1024160061-UCS420/blob/main/UCS420%2C_Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import re


# Q1: Build Your Personalized Knowledge Base
roll_number = input("Enter your college roll number: ").strip()

last_two_digits = roll_number[-2:]

category_map = ["billing", "account", "general"]
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

personalized_entries = []

faq_templates = {
    "billing": {
        "question": "how can i check my payment status",
        "answer": "You can check your payment status from the Billing section.",
        "keywords": "payment status billing transaction"
    },
    "account": {
        "question": "how do i update my registered mobile number",
        "answer": "Go to Account Settings and update your registered mobile number.",
        "keywords": "mobile number update account"
    },
    "general": {
        "question": "where can i get general support",
        "answer": "You can contact the support desk for general assistance.",
        "keywords": "support help assistance"
    }
}

for digit in last_two_digits:
    d = int(digit)
    category = category_map[d % 3]

    entry = faq_templates[category].copy()
    entry["category"] = category

    personalized_entries.append(entry)
all_entries = fixed_entries + personalized_entries

df = pd.DataFrame(all_entries)

print("\n" + "=" * 60)
print("Q1: FINAL 6-ROW KNOWLEDGE BASE")
print("=" * 60)

print(df.to_string(index=False))




Enter your college roll number: 1024160061

Q1: FINAL 6-ROW KNOWLEDGE BASE
                                   question                                                           answer                           keywords category
                     what is the annual fee                                        The annual fee is Rs 500.              fee cost price charge  billing
                      how to reset password                                 Go to Settings > Reset Password.               password reset login  account
                what are your working hours                                        We are open 9 AM to 5 PM.             hours timing open time  general
                      how can i pay the fee                       You can pay via UPI, card, or net banking.                pay payment upi fee  billing
          how can i check my payment status      You can check your payment status from the Billing section. payment status billing transaction  billing
how do 

In [2]:

# Q2: Generate and Score a Hypothesis
def score_query(query, df):
    """
    Takes a query string and returns all matching entries
    ranked by confidence score.
    """

    query_words = set(re.findall(r'\b\w+\b', query.lower()))

    results = []

    for index, row in df.iterrows():

        searchable_text = (
            str(row["question"]).lower()
            + " "
            + str(row["keywords"]).lower()
        )

        entry_words = set(
            re.findall(r'\b\w+\b', searchable_text)
        )


        matched_words = query_words.intersection(entry_words)

        score = len(matched_words)

        if score > 0:
            confidence = score / len(query_words)

            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score,
                "confidence": round(confidence, 2)
            })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results

query = input("\nEnter a FAQ query: ")

results = score_query(query, df)

print("\n" + "=" * 60)
print("Q2: MATCHING RESULTS")
print("=" * 60)

if results:
    for result in results:
        print(f"\nQuestion: {result['question']}")
        print(f"Answer: {result['answer']}")
        print(f"Category: {result['category']}")
        print(f"Score: {result['score']}")
        print(f"Confidence: {result['confidence']}")
else:
    print("No matching FAQ found.")





Enter a FAQ query: index

Q2: MATCHING RESULTS
No matching FAQ found.


In [3]:

# Q3: same_category(category_name, df)
def same_category(category_name, df):
    """
    Returns all questions belonging to the given category.
    """
    return df[df["category"].str.lower() == category_name.lower()]

personalized_category = personalized_entries[0]["category"]

category_result = same_category(
    personalized_category,
    df
)

print("\n" + "=" * 60)
print("Q3: SAME CATEGORY RESULTS")
print("=" * 60)

print(f"Category selected: {personalized_category}")

print(
    category_result[
        ["question", "category"]
    ].to_string(index=False)
)




Q3: SAME CATEGORY RESULTS
Category selected: billing
                         question category
           what is the annual fee  billing
            how can i pay the fee  billing
how can i check my payment status  billing


In [4]:

# Q4: Add a New Keyword and Save CSV
print("\n" + "=" * 60)
print("Q4: ADD NEW KEYWORD")
print("=" * 60)
selected_index = 0

print("Selected FAQ:")
print(df.loc[selected_index, "question"])

new_keyword = input(
    "Enter a new keyword to add: "
).strip()

if new_keyword:

    old_keywords = df.loc[
        selected_index, "keywords"
    ]

    df.loc[
        selected_index, "keywords"
    ] = old_keywords + " " + new_keyword

    print("\nUpdated entry:")
    print(df.loc[selected_index])

filename = f"{roll_number}_faq_data.csv"

df.to_csv(filename, index=False)

print(f"\nUpdated DataFrame saved as: {filename}")



Q4: ADD NEW KEYWORD
Selected FAQ:
what is the annual fee
Enter a new keyword to add: Newkey

Updated entry:
question          what is the annual fee
answer         The annual fee is Rs 500.
keywords    fee cost price charge Newkey
category                         billing
Name: 0, dtype: object

Updated DataFrame saved as: 1024160061_faq_data.csv


In [5]:

# Q5: Count FAQ Entries Per Category
print("\n" + "=" * 60)
print("Q5: FAQ COUNT PER CATEGORY")
print("=" * 60)

category_counts = df.groupby("category").size()

print(category_counts)


# Q6: Handle Ties
def score_query_with_ties(query, df):
    """
    Scores all matching entries.

    If multiple entries have the same highest score,
    all highest-scoring entries are displayed.
    """

    results = score_query(query, df)

    if not results:
        print("\nNo matching entries found.")
        return

    highest_score = results[0]["score"]

    top_matches = [
        result
        for result in results
        if result["score"] == highest_score
    ]

    print("\nQuery:", query)
    print("Highest Score:", highest_score)

    if len(top_matches) > 1:
        print("\nTIE DETECTED!")
        print("All equally matching entries:")

        for result in top_matches:
            print("\nQuestion:", result["question"])
            print("Answer:", result["answer"])
            print("Category:", result["category"])
            print("Score:", result["score"])

    else:
        print("\nUnique highest-scoring match:")

        result = top_matches[0]

        print("\nQuestion:", result["question"])
        print("Answer:", result["answer"])
        print("Category:", result["category"])
        print("Score:", result["score"])




Q5: FAQ COUNT PER CATEGORY
category
account    2
billing    3
general    1
dtype: int64


In [6]:

# Q6 Demonstration 1: Query producing a tie
print("\n" + "=" * 60)
print("Q6: TIE DEMONSTRATION")
print("=" * 60)

score_query_with_ties("fee", df)
# Q6 Demonstration 2: Query without a tie
print("\n" + "-" * 60)
print("Q6: NON-TIE DEMONSTRATION")
print("-" * 60)

score_query_with_ties("reset password", df)


Q6: TIE DEMONSTRATION

Query: fee
Highest Score: 1

TIE DETECTED!
All equally matching entries:

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing
Score: 1

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Score: 1

------------------------------------------------------------
Q6: NON-TIE DEMONSTRATION
------------------------------------------------------------

Query: reset password
Highest Score: 2

Unique highest-scoring match:

Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account
Score: 2
